In [2]:
import pandas as pd
from google.colab import files

print("Por favor, selecciona el archivo CSV original de tu computador:")
uploaded = files.upload()
nombre_archivo = list(uploaded.keys())[0]

df = pd.read_csv(nombre_archivo)
print(f"¡Archivo '{nombre_archivo}' cargado correctamente!")

# 2. SELECCIÓN DE COLUMNAS
columnas_claves = [
    'id', 'region', 'sexo', 'tramo_etario', 'informalidad',
    'ganancia_final', 'tramos_ganancias', 'cine_eme_red',
    'lugar_trabajo', 'financiamiento_inicial', 'motivacion', 'c1_b'
]

col_ganancia = 'ganancia_final' if 'ganancia_final' in df.columns else ('ganancia_estimada' if 'ganancia_estimada' in df.columns else None)
if col_ganancia and col_ganancia != 'ganancia_final':
    columnas_claves[columnas_claves.index('ganancia_final')] = col_ganancia

df_dashboard = df[[c for c in columnas_claves if c in df.columns]].copy()

# 3. MAPEO DE DATOS (basados en el INE)
diccionario_regiones = {
    1: "Tarapacá", 2: "Antofagasta", 3: "Atacama", 4: "Coquimbo",
    5: "Valparaíso", 6: "O'Higgins", 7: "Maule", 8: "Biobío",
    9: "La Araucanía", 10: "Los Lagos", 11: "Aysén", 12: "Magallanes",
    13: "Metropolitana", 14: "Los Ríos", 15: "Arica y Parinacota", 16: "Ñuble"
}
if 'region' in df_dashboard.columns:
    df_dashboard["region"] = df_dashboard["region"].map(diccionario_regiones)

if 'sexo' in df_dashboard.columns:
    df_dashboard["sexo"] = df_dashboard["sexo"].map({1: "Hombre", 2: "Mujer"})

diccionario_edad = {
    1: "15 a 24 años", 2: "25 a 34 años", 3: "35 a 44 años",
    4: "45 a 54 años", 5: "55 a 64 años", 6: "65 años o más"
}
if 'tramo_etario' in df_dashboard.columns:
    df_dashboard["tramo_etario"] = df_dashboard["tramo_etario"].map(diccionario_edad)

if 'informalidad' in df_dashboard.columns:
    df_dashboard["informalidad"] = df_dashboard["informalidad"].map({1.0: "Informal", 0.0: "Formal"})

diccionario_educacion = {
    1: "Sin instrucción / Básica incompleta", 2: "Básica completa / Media incompleta",
    3: "Media completa", 4: "Superior incompleta (IP/CFT/Univ)",
    5: "Superior completa (IP/CFT)", 6: "Superior completa (Universitaria o más)"
}
if 'cine_eme_red' in df_dashboard.columns:
    df_dashboard["cine_eme_red"] = df_dashboard["cine_eme_red"].map(diccionario_educacion)

diccionario_lugar = {
    1: "En su propio hogar", 2: "En el hogar del cliente", 3: "Local/Oficina/Taller establecido",
    4: "En la vía pública / Kiosco", 5: "Vehículo", 6: "No tiene un lugar fijo",
    7: "Predio agrícola", 8: "Obra en construcción", 9: "Otro lugar fijo fuera del hogar"
}
if 'lugar_trabajo' in df_dashboard.columns:
    df_dashboard["lugar_trabajo"] = df_dashboard["lugar_trabajo"].map(diccionario_lugar)

diccionario_finan = {
    1: "Solo ahorros/recursos propios", 2: "Préstamos de familiares/amigos",
    3: "Financiamiento formal (Bancos/Instituciones)", 4: "Subsidios públicos / Fondos del Estado"
}
if 'financiamiento_inicial' in df_dashboard.columns:
    df_dashboard["financiamiento_inicial"] = df_dashboard["financiamiento_inicial"].map(diccionario_finan)

diccionario_motivacion = {
    1: "Tradición familiar", 2: "Oportunidad (Aumentar ingresos/Independencia)",
    3: "Necesidad (Desempleo/Falta de oportunidades)", 4: "Complementar ingresos", 77: "Otra razón"
}
if 'motivacion' in df_dashboard.columns:
    df_dashboard["motivacion"] = df_dashboard["motivacion"].map(diccionario_motivacion)

diccionario_tramos_gan = {
    1: "$0 a $300.000", 2: "$300.001 a $600.000", 3: "$600.001 a $1.000.000",
    4: "$1.000.001 a $1.500.000", 5: "$1.500.001 a $2.500.000", 6: "Más de $2.500.000"
}
if 'tramos_ganancias' in df_dashboard.columns:
    df_dashboard["tramos_ganancias"] = df_dashboard["tramos_ganancias"].map(diccionario_tramos_gan)

# DATOS RAMA ECONÓMICA (CIIU)
diccionario_rama = {
    1: "Agricultura, ganadería, silvicultura y pesca", 2: "Explotación de minas y canteras",
    3: "Industrias manufactureras", 4: "Suministro de electricidad, gas, agua",
    5: "Construcción", 6: "Comercio y reparación de vehículos",
    7: "Transporte y almacenamiento", 8: "Alojamiento y servicios de comidas",
    9: "Información y comunicaciones", 10: "Actividades financieras y de seguros",
    11: "Actividades inmobiliarias", 12: "Actividades profesionales, científicas",
    13: "Servicios administrativos y de apoyo", 14: "Enseñanza",
    15: "Atención de la salud humana", 16: "Actividades artísticas y recreativas",
    17: "Otras actividades de servicios", 18: "Hogares como empleadores",
    19: "Organizaciones extraterritoriales"
}
if 'c1_b' in df_dashboard.columns:
    df_dashboard["c1_b"] = df_dashboard["c1_b"].map(diccionario_rama)


# 4. LIMPIEZA Y RENOMBRES (más identificable)
columnas_limpieza = [c for c in ['region', 'sexo', 'informalidad'] if c in df_dashboard.columns]
df_dashboard = df_dashboard.dropna(subset=columnas_limpieza)

if 'cine_eme_red' in df_dashboard.columns:
    df_dashboard = df_dashboard.rename(columns={'cine_eme_red': 'nivel_educativo'})

if 'c1_b' in df_dashboard.columns:
    df_dashboard = df_dashboard.rename(columns={'c1_b': 'rama_economica'})


# 5. EXPORTAR EL ARCHIVO FINAL
nombre_salida = "dataset_limpio.csv"
df_dashboard.to_csv(nombre_salida, index=False)

#DESCARGA AUTOMATICA
files.download(nombre_salida)

Por favor, selecciona el archivo CSV original de tu computador:


Saving Base_de_Datos_Full_VIII_EME_LIMPIA.csv to Base_de_Datos_Full_VIII_EME_LIMPIA.csv
¡Archivo 'Base_de_Datos_Full_VIII_EME_LIMPIA.csv' cargado correctamente!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>